여기서는 LLM이 자신이 하고 있는 일을 잘 하고 있는지 셀프로 검증하는 self-rag를 학습한다.

In [1]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')
vector_store = Chroma(
    embedding_function=embeddings,
    collection_name='income_tax_collection',
    persist_directory='./income_tax_collection'
)

retriever = vector_store.as_retriever(search_kwargs={'k': 3})


In [2]:
# START -> RETRIEVE -> GENERATE -> END

from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    query: str
    context: List[Document]
    answer: str

graph_builder = StateGraph(AgentState)

In [3]:
def retrieve(state: AgentState):
    query = state['query']
    docs = retriever.invoke(query)
    return {'context': docs}    

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [5]:
from langchain import hub

generate_prompt = hub.pull('rlm/rag-prompt')
generate_llm = ChatOpenAI(model='gpt-4o', max_tokens=100)
def generate(state: AgentState) -> AgentState:
    context = state['context']
    query = state['query']
    rag_chain = generate_prompt | llm
    response = rag_chain.invoke({'question': query, 'context': context})
    return {'answer': response.content}

/Users/jys/code/study/study-llm-agent/venv/lib/python3.11/site-packages/langsmith/client.py:278: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [6]:
from langchain import hub
from typing import Literal

# 질문과 document의 연관성이 얼마나 되는지 확인하는 rag-document-relevance 프롬프트
doc_relevance_prompt = hub.pull('langchain-ai/rag-document-relevance')

# self-rag에서는 END를 호출을 해야 하는데, END는 노드가 아니기 떄문에, Literal을 해도 에러가 발생한다. 따라서 엣지에서 처리해야 한다.
def check_doc_relevance(state: AgentState) -> Literal['relevant', 'irrelevant']:
    context = state['context']
    query = state['query']
    doc_relevance_chain = doc_relevance_prompt | llm
    response = doc_relevance_chain.invoke({'question': query, 'documents': context})
    if response['Score'] == 1:
        return 'relevant'
    return 'irrelevant'

/Users/jys/code/study/study-llm-agent/venv/lib/python3.11/site-packages/langsmith/client.py:278: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

dictionary = ['사람과 관련된 표현 -> 거주자']

rewrite_prompt = PromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    사전: {dictionary}
    질문: {{query}}
"""
)

def rewrite(state: AgentState):
    query = state['query']
    rewrite_chain = rewrite_prompt | llm | StrOutputParser()
    response = rewrite_chain.invoke({'query': query})
    return {'query': response}


In [8]:
from langchain import hub

hallucination_prompt = hub.pull('langchain-ai/rag-answer-hallucination') -> Literal['hallucinated', 'not hallucinated']

def check_hallucination(state: AgentState):
    answer = state['answer']
    context = state['context']
    hallucination_chain = hallucination_prompt | llm
    response = hallucination_chain.invoke({'student_answer': answer, 'documents': context})
    return response

/Users/jys/code/study/study-llm-agent/venv/lib/python3.11/site-packages/langsmith/client.py:278: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [14]:
from langchain import hub

helpfulness_prompt = hub.pull('langchain-ai/rag-answer-helpfulness')

def check_helpfulness(state: AgentState):
    query = state['query']
    answer = state['answer']
    helpfulness_chain = helpfulness_prompt | llm
    response = helpfulness_chain.invoke({'question': query, 'student_answer': answer})
    return response


/Users/jys/code/study/study-llm-agent/venv/lib/python3.11/site-packages/langsmith/client.py:278: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
# 할루시네이션을 검증하는 prompt의 테스트

query = '연봉 5천만원인 거주자의 소득세는 얼마인가요?'

context = retriever.invoke(query)
generate_state = {'query': query, 'context': context}
answer = generate(generate_state)

# langchain 할루시네이션 프롬프트를 사용 시 할루시네이션이 발생했다고 판단한다.
# 문서에 있는 내용을 근거로 해서 답변을 해서 답변이 길어서로 유추된다.
# langchain에서 제공하는 prompt를 사용하지 않고, 직접 prompt를 수정해서 사용해야 할것 같다.
hallucination_state = {'answer': answer, 'context': context}

check_hallucination(hallucination_state)

{'Score': 0,
 'Explanation': "The student's answer contains an error in the calculation and application of the tax rate. According to the given tax brackets:\n\n1. For income up to 1,400만원, the tax is 84만원 plus 15% of the amount over 1,400만원.\n2. For income up to 8,800만원, the tax is 624만원 plus 24% of the amount over 5,000만원.\n\nThe student claims that a resident with an annual income of 5,000만원 pays 624만원 in tax because it does not exceed 5,000만원, applying the basic amount for the 5,000만원 to 8,800만원 bracket. However, this ignores the requirement to add the 24% of the amount exceeding 5,000만원 once it goes beyond 1,400만원 but doesn't reach the next tier.\n\nTherefore, the calculated tax amount (624만원) for a 5,000만원 income is incorrect and does not fully adhere to the tax structure detailed in the facts. Thus, the answer is not fully grounded in the facts."}

In [15]:
# 할루시네이션을 검증하는 prompt의 테스트

query = '연봉 5천만원인 거주자의 소득세는 얼마인가요?'

context = retriever.invoke(query)
generate_state = {'query': query, 'context': context}
answer = generate(generate_state)

helpfulness_state = {'query': query, 'answer': answer}

check_helpfulness(helpfulness_state)

{'Score': 1,
 'Explanation': "The student's answer is concise and directly answers the question. It states that the income tax for a resident with an income of 50 million won is 6.24 million won. Additionally, it provides a brief explanation of how this figure is computed, which is relevant to the question being asked."}